In [2]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# Forzar la descarga de los recursos necesarios de NLTK para este entorno
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

# Cargar el dataset desde la carpeta data del repositorio
df = pd.read_csv('../data/libros.csv')

# Verificar la carga exitosa de los 150 libros
print(f"Total de documentos cargados en el corpus: {len(df)}")
display(df.head(2))

# Configuración del preprocesamiento diferencial exigido por la cátedra
stop_words = set(stopwords.words('spanish'))

def limpiar_texto(texto):
    if pd.isna(texto):
        return ""
    # Pasar a minúsculas
    texto = str(texto).lower()
    # Eliminar puntuación y números
    texto = re.sub(r'[^\w\s]', '', texto)
    # Tokenizar y eliminar stopwords
    tokens = word_tokenize(texto)
    tokens_limpios = [t for t in tokens if t not in stop_words]
    return " ".join(tokens_limpios)

# 1. Versión Cruda (Para SBERT y modelos de oración)
df['texto_crudo'] = df['titulo'].fillna('') + ". " + df['sinopsis'].fillna('')

# 2. Versión Limpia (Para TF-IDF y Word2Vec propio)
df['texto_limpio'] = df['texto_crudo'].apply(limpiar_texto)

print("\n--- Verificación de transformaciones ---")
print("CRUDO:", df['texto_crudo'].iloc[0][:120])
print("LIMPIO:", df['texto_limpio'].iloc[0][:120])

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\juanm\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\juanm\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


Total de documentos cargados en el corpus: 150


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\juanm\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


,titulo,autores,generos,serie,sinopsis,url_libro,categoria_origen,fecha_extraccion
0,NaN,Verónica Cervilla,"Drama, Histórico, Novela",NaN,NaN,https://ww3.lectulandia.co/book/cuando-callan-...,Drama,2026-09-15
1,NaN,Martín Casariego,"Drama, Novela",NaN,NaN,https://ww3.lectulandia.co/book/como-los-pajar...,Drama,2026-09-15



--- Verificación de transformaciones ---
CRUDO: . 
LIMPIO: 


In [3]:
# Ver los nombres reales de las columnas de tu dataframe para estar seguros
print("Columnas disponibles en el CSV:", df.columns.tolist())

# Si la columna del título o la sinopsis se llama distinto, las ajustamos acá.
# Por lo general, probemos armando el texto crudo usando el autor y los géneros si la sinopsis vino vacía:
df['texto_crudo'] = "Libro de " + df['autores'].fillna('desconocido') + ". Géneros: " + df['generos'].fillna('drama')

# Volvemos a aplicar la limpieza diferencial
df['texto_limpio'] = df['texto_crudo'].apply(limpiar_texto)

print("\n--- Texto Crudo Corregido ---")
print(df['texto_crudo'].iloc[0])
print("\n--- Texto Limpio Corregido ---")
print(df['texto_limpio'].iloc[0])

# --- ENTRENAMIENTO DE WORD2VEC (Exigido por la cátedra) ---
from gensim.models import Word2Vec

# Preparamos los documentos tokenizados en forma de lista de listas (palabras)
corpus_tokens = [doc.split() for doc in df['texto_limpio']]

# Entrenamos nuestro propio modelo Word2Vec desde cero con el corpus de Drama
# vector_size=100 (tamaño del vector), window=5 (ventana de contexto), min_count=1 (para corpus chico de 150 libros)
w2v_model = Word2Vec(sentences=corpus_tokens, vector_size=100, window=5, min_count=1, workers=4, epochs=50)

print(f"\n¡Modelo Word2Vec entrenado con éxito!")
print(f"Vocabulario total aprendido: {len(w2v_model.wv.key_to_index)} palabras únicas.")

# Probamos una búsqueda de palabras similares si existen en el corpus
if 'amor' in w2v_model.wv:
    print("\nPalabras más similares a 'amor' según nuestro modelo de Drama:")
    print(w2v_model.wv.most_similar('amor', topn=3))

Columnas disponibles en el CSV: ['titulo', 'autores', 'generos', 'serie', 'sinopsis', 'url_libro', 'categoria_origen', 'fecha_extraccion', 'texto_crudo', 'texto_limpio']

--- Texto Crudo Corregido ---
Libro de Verónica Cervilla. Géneros: Drama, Histórico, Novela

--- Texto Limpio Corregido ---
libro verónica cervilla géneros drama histórico novela

¡Modelo Word2Vec entrenado con éxito!
Vocabulario total aprendido: 322 palabras únicas.


In [4]:
from sentence_transformers import SentenceTransformer
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# 1. Cargamos un modelo SBERT liviano y multilingüe optimizado para similitud semántica
print("Cargando el modelo SBERT (esto puede demorar unos segundos la primera vez)...")
sbert_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

# 2. Generamos los embeddings de todo nuestro corpus de libros usando el texto crudo
# (SBERT prefiere el texto crudo porque entiende la estructura y puntuación natural)
print("Generando vectores de embeddings para los 150 libros...")
corpus_embeddings = sbert_model.encode(df['texto_crudo'].tolist(), show_progress_bar=True)

print(f"¡Embeddings generados con éxito!")
print(f"Dimensiones de la matriz de vectores: {corpus_embeddings.shape}") # Debería ser (150, 384)

# 3. Probamos una consulta de prueba en lenguaje natural contra nuestros libros
consulta_prueba = "historias de amor trágico y desamor"
vector_consulta = sbert_model.encode([consulta_prueba])

# Calculamos la similitud de coseno entre la consulta y todos los libros del dataset
similitudes = cosine_similarity(vector_consulta, corpus_embeddings)[0]

# Obtenemos los índices de los 3 libros más similares
top_3_indices = np.argsort(similitudes)[::-1][:3]

print(f"\n--- Resultados para la consulta: '{consulta_prueba}' ---")
for i in top_3_indices:
    print(f"Puntaje: {similitudes[i]:.4f} | Autor/Género: {df['autores'].iloc[i]} ({df['generos'].iloc[i]})")

e:\TUIA\2do Año\2do cuatrimestre\Procesamiento del lenguaje natural\NLP_Ayala_Cerana_Colombo_Ibarbia\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Cargando el modelo SBERT (esto puede demorar unos segundos la primera vez)...


e:\TUIA\2do Año\2do cuatrimestre\Procesamiento del lenguaje natural\NLP_Ayala_Cerana_Colombo_Ibarbia\venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\juanm\.cache\huggingface\hub\models--sentence-transformers--paraphrase-multilingual-MiniLM-L12-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loa

Generando vectores de embeddings para los 150 libros...


Batches: 100%|██████████| 5/5 [00:02<00:00,  2.41it/s]


¡Embeddings generados con éxito!
Dimensiones de la matriz de vectores: (150, 384)

--- Resultados para la consulta: 'historias de amor trágico y desamor' ---
Puntaje: 0.6762 | Autor/Género: Janne Teller (Drama, Novela, Psicológico, Romántico)
Puntaje: 0.6118 | Autor/Género: Carolina Casado (Drama, Novela, Romántico)
Puntaje: 0.6111 | Autor/Género: Elia Barceló (Drama, Intriga, Psicológico, Relato)


In [5]:
import json

# 1. Cargamos nuestro archivo queries.json de evaluación
with open('../queries.json', 'r', encoding='utf-8') as f:
    queries_eval = json.load(f)

print(f"Se cargaron {len(queries_eval)} consultas para evaluar.\n")

# 2. Función rápida para evaluar Precision@3 usando SBERT
def evaluar_buscador_sbert(queries, sbert_model, corpus_embeddings, df, k=3):
    resultados_eval = []
    
    for item in queries:
        q_id = item['id']
        q_text = item['query']
        libros_esperados = item['libros_relevantes']
        
        # Generar embedding de la consulta
        vec_q = sbert_model.encode([q_text])
        sims = cosine_similarity(vec_q, corpus_embeddings)[0]
        
        # Obtener los top k índices más similares
        top_indices = np.argsort(sims)[::-1][:k]
        
        # Extraer los slugs o identificadores de los libros predichos por el modelo
        # (Usamos el slug extraído de la url como clave de comparación)
        preds = []
        for idx in top_indices:
            url = df['url_libro'].iloc[idx]
            slug = str(url).strip('/').split('/')[-1]
            preds.append(slug)
            
        # Calcular aciertos (cuántos de los predichos están en los relevantes)
        aciertos = len(set(preds).intersection(set(libros_esperados)))
        precision = aciertos / k if k > 0 else 0
        
        resultados_eval.append({
            "id": q_id,
            "query": q_text,
            "precision_at_k": precision,
            "predicciones": preds,
            "esperados": libros_esperados
        })
        
    return resultados_eval

# Ejecutamos la evaluación
eval_resultados = evaluar_buscador_sbert(queries_eval, sbert_model, corpus_embeddings, df, k=3)

# Mostramos un resumen de los resultados por consola
print("--- RESUMEN DE EVALUACIÓN (Precision@3) ---")
precisions = []
for res in eval_resultados:
    print(f"Query {res['id']}: '{res['query']}' -> Precision@3: {res['precision_at_k']:.2f}")
    precisions.append(res['precision_at_k'])

print(f"\nPromedio general de Precision@3 del modelo SBERT: {np.mean(precisions):.2f}")

Se cargaron 10 consultas para evaluar.

--- RESUMEN DE EVALUACIÓN (Precision@3) ---
Query 1: 'historias de amor trágico y desamor en la vida real' -> Precision@3: 0.00
Query 2: 'dramas familiares y secretos oscuros' -> Precision@3: 0.00
Query 3: 'relatos sobre la superación del dolor y la pérdida' -> Precision@3: 0.00
Query 4: 'conflictos de clases sociales y romances prohibidos' -> Precision@3: 0.00
Query 5: 'historias ambientadas en épocas de guerra y sufrimiento' -> Precision@3: 0.00
Query 6: 'tragedias juveniles y decisiones difíciles' -> Precision@3: 0.00
Query 7: 'relatos de culpa, redención y juicios morales' -> Precision@3: 0.00
Query 8: 'vínculos familiares rotos y reconciliación' -> Precision@3: 0.00
Query 9: 'historias de profunda melancolía y soledad' -> Precision@3: 0.00
Query 10: 'lucha contra la enfermedad y la adversidad extrema' -> Precision@3: 0.00

Promedio general de Precision@3 del modelo SBERT: 0.00


In [ ]:
import psycopg2
import numpy as np
from pgvector.psycopg2 import register_vector

try:
    # Conexión usando los parámetros individuales que te muestra Supabase abajo
    conn = psycopg2.connect(
        host="db.paynmaeteawtpdqcpazt.supabase.co",
        database="postgres",
        user="postgres",
        password="TU_CONTRASEÑA_REAL",  # Poné tu contraseña real acá
        port="5432"
    )
    
    # Registrar el tipo vector en esta conexión
    register_vector(conn)
    
    cursor = conn.cursor()
    print("¡Conexión exitosa a Supabase!")

    # 1. Habilitar la extensión pgvector
    cursor.execute("CREATE EXTENSION IF NOT EXISTS vector;")
    
    # 2. Crear la tabla para almacenar los libros y sus embeddings
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS libros_vectoriales (
            id SERIAL PRIMARY KEY,
            titulo TEXT,
            autores TEXT,
            generos TEXT,
            sinopsis TEXT,
            embedding vector(384)
        );
    """)
    conn.commit()
    print("Tabla 'libros_vectoriales' creada o verificada con éxito.")

    # 3. Limpiar registros previos para evitar duplicados
    cursor.execute("DELETE FROM libros_vectoriales;")
    
    # 4. Insertar los datos y vectores del DataFrame
    for idx, row in df.iterrows():
        vec_list = corpus_embeddings[idx].tolist()
        
        cursor.execute("""
            INSERT INTO libros_vectoriales (titulo, autores, generos, sinopsis, embedding)
            VALUES (%s, %s, %s, %s, %s);
        """, (
            str(row.get('titulo', '')),
            str(row.get('autores', '')),
            str(row.get('generos', '')),
            str(row.get('sinopsis', '')),
            vec_list
        ))
        
    conn.commit()
    print(f"¡Se insertaron exitosamente los {len(df)} libros con sus vectores en Supabase!")

except Exception as e:
    print(f"Error al conectar o insertar en la base de datos: {e}")

finally:
    if 'cursor' in locals() and cursor:
        cursor.close()
    if 'conn' in locals() and conn:
        conn.close()

Error al conectar o insertar en la base de datos: could not translate host name "db.paynmaeteawtpdqcpazt.supabase.co" to address: Name or service not known

